# Short Term Memory: LangGraph Checkpoints in MongoDB

**Short-term memory** is the temporary store of information an agent holds while performing a task or during an interaction. It keeps track of what is happening in the current conversation, session, or reasoning process — but is not meant to last beyond that session unless deliberately saved.

The presentation defines two forms of short-term memory:

- **Working memory** — what is actively in the LLM's context window right now
- **Session memory** — the history of the current conversation, stored externally so it can be replayed into the context window on every turn

This notebook implements both using **LangGraph** and **MongoDB**. LangGraph's `MongoDBSaver` checkpointer persists the full agent state (messages, tool calls, intermediate steps) in MongoDB at every step. Each conversation is identified by a `thread_id`, enabling session isolation and restoration.

```
User → [Message + thread_id] → LangGraph Agent
                                      ↕
                              MongoDB (checkpoints)
                                      ↕
                              State restored on next turn
```

> **Scenario:** A travel assistant that helps users find Airbnb-style listings. It remembers everything said in the current session — city preferences, budget, room type — without the user needing to repeat themselves.

In [ ]:
import { MongoClient } from 'mongodb';
import { ChatAnthropic } from '@langchain/anthropic';
import { createReactAgent } from '@langchain/langgraph/prebuilt';
import { MongoDBSaver } from '@langchain/langgraph-checkpoint-mongodb';
import { tool } from '@langchain/core/tools';
import { z } from 'zod';

// ← Set ANTHROPIC_API_KEY as a Codespace secret, or paste it here as a fallback
const ANTHROPIC_API_KEY = process.env.ANTHROPIC_API_KEY ?? 'sk-ant-...';

const client = new MongoClient(process.env.MONGODB_URI!, { appName: 'devrel-workshop-agentmemory-langchain' });
await client.connect();
const db       = client.db('agent_memory_lab');
const listings = db.collection('listings');

console.log('Connected. Listings:', await listings.countDocuments());

## Seeding listings into the memory lab database

The seed script populates `voyage_lab.listings`. Here we copy those listings into `agent_memory_lab.listings` so this notebook is self-contained.

In [ ]:
const sourceColl = client.db('voyage_lab').collection('listings');
const count = await listings.countDocuments();

if (count === 0) {
  const docs = await sourceColl.find({}).toArray();
  await listings.insertMany(docs);
  console.log(`Copied ${docs.length} listings into agent_memory_lab.listings`);
} else {
  console.log(`listings already present: ${count}`);
}

## Defining the listing search tool

The agent has one tool: `search_listings`. It queries MongoDB for listings matching city, max price, and room type. The agent decides when to call it based on what the user says.

In [ ]:
const searchListings = tool(
  async ({ city, maxPrice, roomType }: { city: string; maxPrice?: number; roomType?: string }) => {
    const filter: Record<string, unknown> = { 'address.market': { $regex: city, $options: 'i' } };
    if (maxPrice) filter['price'] = { $lte: maxPrice };
    if (roomType) filter['room_type'] = { $regex: roomType, $options: 'i' };

    const results = await listings
      .find(filter, { projection: { name: 1, price: 1, room_type: 1, bedrooms: 1, amenities: 1 } })
      .limit(3)
      .toArray();

    if (results.length === 0) return 'No listings found matching those criteria.';

    return results
      .map(r => `• ${r.name} — $${r.price}/night (${r.room_type}, ${r.bedrooms}BR)`)
      .join('\n');
  },
  {
    name: 'search_listings',
    description: 'Search Airbnb-style listings by city, max price per night, and room type.',
    schema: z.object({
      city:     z.string().describe('City name, e.g. Barcelona'),
      maxPrice: z.number().optional().describe('Maximum price per night in USD'),
      roomType: z.string().optional().describe('Room type: Entire home/apt, Private room, Shared room'),
    }),
  }
);

console.log('Tool ready:', searchListings.name);

## Creating the agent with MongoDB checkpointing

`MongoDBSaver` is LangGraph's built-in checkpointer for MongoDB. It writes the full agent state to a `checkpoints` collection after every step. When the same `thread_id` is used on the next turn, the state is automatically restored — giving the agent session memory.

In [ ]:
const llm = new ChatAnthropic({
  model: 'claude-haiku-4-5-20251001',
  apiKey: ANTHROPIC_API_KEY,
});

const checkpointer = new MongoDBSaver({ client, dbName: 'agent_memory_lab' });

const agent = createReactAgent({
  llm,
  tools: [searchListings],
  checkpointSaver: checkpointer,
  prompt:
    'You are a helpful travel assistant that finds Airbnb-style accommodations. ' +
    'Be concise. When the user mentions a city or budget, search for listings. ' +
    'Remember everything the user has told you in this conversation.',
});

console.log('Agent created with MongoDBSaver checkpointer.');

## First turn — the agent responds and saves state

Every invocation uses a `thread_id` inside `configurable`. LangGraph stores the resulting state in MongoDB under that thread.

In [ ]:
const THREAD_A = 'travel-session-alice-001';

const turn1 = await agent.invoke(
  { messages: [{ role: 'user', content: 'Hi! I am looking for a place in Barcelona under $150/night.' }] },
  { configurable: { thread_id: THREAD_A } }
);

const lastMessage = turn1.messages[turn1.messages.length - 1];
console.log('Agent:', lastMessage.content);

## Inspecting the checkpoint in MongoDB

LangGraph wrote the agent's state to `agent_memory_lab.checkpoints`. Let's look at what was persisted.

In [ ]:
const checkpointsColl = db.collection('checkpoints');

const checkpoint = await checkpointsColl.findOne(
  { thread_id: THREAD_A },
  { sort: { checkpoint_id: -1 } }
);

console.log('thread_id:    ', checkpoint?.thread_id);
console.log('checkpoint_id:', checkpoint?.checkpoint_id);
console.log('Fields stored:', Object.keys(checkpoint ?? {}).join(', '));

const allForThread = await checkpointsColl.countDocuments({ thread_id: THREAD_A });
console.log(`Total checkpoints for thread "${THREAD_A}":`, allForThread);

## Second turn — the agent remembers

The user asks a follow-up without repeating the city or budget. LangGraph rehydrates the full conversation from MongoDB before calling the LLM.

In [ ]:
const turn2 = await agent.invoke(
  { messages: [{ role: 'user', content: 'Do any of those have a private room option? I prefer not sharing.' }] },
  { configurable: { thread_id: THREAD_A } }
);

const reply2 = turn2.messages[turn2.messages.length - 1];
console.log('Agent:', reply2.content);

## Third turn — building further context

The user adds a new constraint. The agent accumulates preferences across turns without losing earlier context.

In [ ]:
const turn3 = await agent.invoke(
  { messages: [{ role: 'user', content: 'Actually, I need at least 2 bedrooms. We are two people traveling together.' }] },
  { configurable: { thread_id: THREAD_A } }
);

const reply3 = turn3.messages[turn3.messages.length - 1];
console.log('Agent:', reply3.content);

const totalCheckpoints = await checkpointsColl.countDocuments({ thread_id: THREAD_A });
console.log(`\nCheckpoints after 3 turns: ${totalCheckpoints}`);

## Thread isolation — a different session starts fresh

A new `thread_id` creates a completely independent session. The agent has no knowledge of the Barcelona conversation.

In [ ]:
const THREAD_B = 'travel-session-bob-001';

const threadB = await agent.invoke(
  { messages: [{ role: 'user', content: 'Do any of those have a private room option?' }] },
  { configurable: { thread_id: THREAD_B } }
);

const replyB = threadB.messages[threadB.messages.length - 1];
console.log('Agent (new thread):', replyB.content);
console.log('\n→ The agent asks for clarification because it has no prior context in this thread.');

## Session restoration — resuming after disconnect

Even if we create a brand-new agent instance, supplying the same `thread_id` restores the full conversation from MongoDB. This simulates a user returning after closing the browser — their session is intact.

In [ ]:
const freshCheckpointer = new MongoDBSaver({ client, dbName: 'agent_memory_lab' });
const freshAgent = createReactAgent({
  llm,
  tools: [searchListings],
  checkpointSaver: freshCheckpointer,
  prompt:
    'You are a helpful travel assistant that finds Airbnb-style accommodations. ' +
    'Be concise. When the user mentions a city or budget, search for listings. ' +
    'Remember everything the user has told you in this conversation.',
});

const restored = await freshAgent.invoke(
  { messages: [{ role: 'user', content: 'Can you also check if any of those have WiFi?' }] },
  { configurable: { thread_id: THREAD_A } }
);

const restoredReply = restored.messages[restored.messages.length - 1];
console.log('Restored agent reply:', restoredReply.content);
console.log('\n→ The fresh agent instance knows about Barcelona and the previous preferences.');

## What the checkpoint document looks like

Let's inspect the structure of a checkpoint to understand what LangGraph stores on MongoDB's behalf.

In [ ]:
const latestCheckpoint = await checkpointsColl.findOne(
  { thread_id: THREAD_A },
  { sort: { checkpoint_id: -1 } }
);

console.log('Checkpoint document keys:', Object.keys(latestCheckpoint ?? {}).join(', '));
console.log('thread_id:    ', latestCheckpoint?.thread_id);
console.log('checkpoint_id:', latestCheckpoint?.checkpoint_id);
console.log('parent_id:    ', latestCheckpoint?.parent_checkpoint_id ?? 'none');

const writesCol = db.collection('checkpoint_writes');
const writesCount = await writesCol.countDocuments({ thread_id: THREAD_A });
console.log(`\ncheckpoint_writes for thread "${THREAD_A}": ${writesCount}`);

In [ ]:
await checkpointsColl.deleteMany({ thread_id: { $in: [THREAD_A, THREAD_B] } });
await db.collection('checkpoint_writes').deleteMany({ thread_id: { $in: [THREAD_A, THREAD_B] } });
await client.close();
console.log('Done.');